Zadanie 1: WroFleet - Polowanie na Anomalie (2 pkt)
Scenariusz: Firma “WroFleet” monitoruje flotę 100 autobusów. Czujniki czasem podają błędne odczyty (awaria sensora), ale czasem prawdziwe ekstremalne wartości (przegrzanie silnika). Mu-
sisz odróżnić błędy od prawdziwych anomalii.

Dane: Użyj danych pzz_fleet_data_full.csv (10,000 odczytów z 5 czujników, 2% błędów
sensorów, 1% prawdziwych anomalii) oraz pzz_fleet_physical_limits.json.

In [9]:
import numpy as np
import pandas as pd
import json
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

df = pd.read_csv("pzz_fleet_data_full.csv")

with open("pzz_fleet_physical_limits.json", "r", encoding="utf-8") as f:
    limits = json.load(f)

df.head()

sensor_cols = ["temperature","vibration","speed","fuel_consumption","load"]

Detekcja outlierów: Zastosuj 3 metody:
• IQR (1.5 × IQR)
• Z-score (|z| > 3)
• Isolation Forest (contamination=0.03)
Ile outlierów wykrywa każda metoda? Czy się pokrywają?

In [10]:
def outliers_iqr(df, cols, k=1.5):
    q1 = df[cols].quantile(0.25)
    q3 = df[cols].quantile(0.75)
    iqr = q3 - q1
    lo = q1 - k * iqr
    hi = q3 + k * iqr
    mask = (df[cols] < lo) | (df[cols] > hi)

    return mask.any(axis=1)

def outliers_zscore(df, cols, thr = 3.0):
    mu = df[cols].mean()
    sig = df[cols].std(ddof=0).replace(0, np.nan)
    z = (df[cols]-mu)/sig
    mask = z.abs() > thr

    return mask.any(axis=1)

def outliers_isoforest(df, cols, contamination=0.03, seed=42):
    X = df[cols].copy()
    X = X.fillna(X.median(numeric_only=True))

    iso = IsolationForest(
        contamination=contamination,
        random_state=seed,
        n_estimators=300,
        n_jobs=1
    )
    pred = iso.fit_predict(X)
    return pred == -1

In [11]:
mask_iqr = outliers_iqr(df, sensor_cols, k=1.5)
mask_z   = outliers_zscore(df, sensor_cols, thr=3.0)
mask_if  = outliers_isoforest(df, sensor_cols, contamination=0.03, seed=42)

S_iqr = set(df.index[mask_iqr])
S_z   = set(df.index[mask_z])
S_if  = set(df.index[mask_if])

print("Liczba outlierow")
print("IQR:             ", len(S_iqr))
print("Z-score:         ", len(S_z))
print("IsolationForest: ", len(S_if))

print("\nPrzeciecia")
print("IQR ∩ Z-score:              ", len(S_iqr & S_z))
print("IQR ∩ IsolationForest:      ", len(S_iqr & S_if))
print("Z-score ∩ IsolationForest:  ", len(S_z & S_if))
print("IQR ∩ Z ∩ IF:               ", len(S_iqr & S_z & S_if))

print("\nUnikalne")
print("Tylko IQR:             ", len(S_iqr - (S_z | S_if)))
print("Tylko Z-score:         ", len(S_z   - (S_iqr | S_if)))
print("Tylko IsolationForest: ", len(S_if  - (S_iqr | S_z)))


Liczba outlierow
IQR:              484
Z-score:          210
IsolationForest:  300

Przeciecia
IQR ∩ Z-score:               210
IQR ∩ IsolationForest:       300
Z-score ∩ IsolationForest:   210
IQR ∩ Z ∩ IF:                210

Unikalne
Tylko IQR:              184
Tylko Z-score:          0
Tylko IsolationForest:  0


In [12]:
def physical_mask(df_part):
    m = pd.Series(False, index=df_part.index)
    for c in sensor_cols:
        lo, hi = limits[c] 
        m |= (df_part[c] < lo) | (df_part[c] > hi)
    return m

sensor_mask = physical_mask(df)

mask_out_any = mask_iqr | mask_z | mask_if
mask_sensor_error = mask_out_any & sensor_mask
mask_true_anomaly = mask_out_any & ~sensor_mask

print("Podzial")
print("Outliery:           ", int(mask_out_any.sum()))
print("Bledy sensora: ", int(mask_sensor_error.sum()))
print("Prawdziwe anomalie:", int(mask_true_anomaly.sum()))
print("Poza limitami:       ", int(sensor_mask.sum()))

cols_show = ["bus_id", "timestamp"] + sensor_cols
print("\nPrzyklady bledow sensora:")
print(df.loc[mask_sensor_error, cols_show].head())

print("\nPrzyklady prawdziwych anomalii:")
print(df.loc[mask_true_anomaly, cols_show].head())

Podzial
Outliery:            484
Bledy sensora:  200
Prawdziwe anomalie: 284
Poza limitami:        200

Przyklady bledow sensora:
     bus_id            timestamp  temperature  vibration   speed  \
20       76  2024-09-19 22:51:00       -29.12     242.94  -10.47   
67        4  2024-04-02 12:42:00      -999.00     -13.12  -34.09   
213      44  2024-02-15 13:19:00       -41.54     467.63  325.68   
257      71  2024-04-18 05:11:00      -999.00     -39.98  -16.49   
283      77  2024-12-18 20:54:00       -11.46     -65.16  -46.61   

     fuel_consumption    load  
20             -25.76  169.08  
67             -16.21  -33.86  
213            -17.49  -10.15  
257            -22.40  -14.21  
283             -9.24  183.96  

Przyklady prawdziwych anomalii:
     bus_id            timestamp  temperature  vibration  speed  \
17       27  2024-07-03 13:33:00        98.18      40.81  15.03   
25       92  2024-04-05 03:54:00        97.56      48.49  35.34   
81       36  2024-07-16 13:54:00   

Strategie obsługi: Zaimplementuj 3 strategie:
• Usunięcie outlierów
• Winsoryzacja (clip do 1./99. percentyla)
• Oznaczenie jako osobna kategoria “anomalia”
Wytrenuj model predykcji awarii z każdą strategią. Która daje najlepszy F1?

In [13]:
y = df["is_failure"].astype(int)

ts = pd.to_datetime(df["timestamp"])
X_base = df[["bus_id"] + sensor_cols].copy()
X_base["hour"] = ts.dt.hour
X_base["dayofweek"] = ts.dt.dayofweek
X_base["month"] = ts.dt.month

mask_out_any = mask_iqr | mask_z | mask_if
sensor_mask  = physical_mask(df)

X_train, X_test, y_train, y_test, out_train, out_test, phys_train, phys_test = train_test_split(
    X_base, y, mask_out_any, sensor_mask,
    test_size=0.25, random_state=42, stratify=y
)

clf = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced"))
])

In [ ]:
def eval_f1(Xtr, ytr, Xte, yte, name):
    clf.fit(Xtr, ytr)
    pred = clf.predict(Xte)
    f1 = f1_score(yte, pred)
    print(f"{name:24s} F1 = {f1:.4f}")
    return f1

keep = ~phys_train

f1_remove = eval_f1(
    X_train.loc[keep], y_train.loc[keep],
    X_test, y_test,
    "remove_sensor_errors"
)

def winsorize(Xtr, Xte, cols, p1=0.01, p2=0.99):
    lo = Xtr[cols].quantile(p1)
    hi = Xtr[cols].quantile(p2)
    Xtr2 = Xtr.copy()
    Xte2 = Xte.copy()
    Xtr2[cols] = Xtr2[cols].clip(lower=lo, upper=hi, axis=1)
    Xte2[cols] = Xte2[cols].clip(lower=lo, upper=hi, axis=1)
    return Xtr2, Xte2

Xtr_w, Xte_w = winsorize(X_train, X_test, sensor_cols, 0.01, 0.99)
f1_wins = eval_f1(Xtr_w, y_train, Xte_w, y_test, "winsorize_1_99")

Xtr_f = X_train.copy()
Xte_f = X_test.copy()

Xtr_f["is_outlier"] = out_train.astype(int).values
Xte_f["is_outlier"] = out_test.astype(int).values

Xtr_f["is_physical_impossible"] = phys_train.astype(int).values
Xte_f["is_physical_impossible"] = phys_test.astype(int).values

f1_flag = eval_f1(Xtr_f, y_train, Xte_f, y_test, "flag_as_features")

remove_sensor_errors     F1 = 0.6757
winsorize_1_99           F1 = 1.0000
flag_as_features         F1 = 0.8929


In [30]:
def production_pipeline(df_part):
    m_iqr = outliers_iqr(df_part, sensor_cols, k=1.5)
    m_z   = outliers_zscore(df_part, sensor_cols, thr=3.0)
    m_if  = outliers_isoforest(df_part, sensor_cols, contamination=0.03, seed=42)
    out_any = m_iqr | m_z | m_if

    phys = physical_mask(df_part)
    sensor_error = out_any & phys
    true_anomaly = out_any & ~phys

    decision = pd.Series("keep", index=df_part.index)
    alert = pd.Series("", index=df_part.index)

    decision[sensor_error] = "remove_or_impute"
    alert[sensor_error] = "alert_sensor_fault"

    decision[true_anomaly] = "keep"
    alert[true_anomaly] = "alert_true_anomaly"

    return decision, alert, sensor_error, true_anomaly

dec, al, se, ta = production_pipeline(df)

print("licznosci")
print("sensor_error:", int(se.sum()))
print("true_anomaly:", int(ta.sum()))
print("\nPrzyklady decyzji:")
print(pd.DataFrame({"decision": dec, "alert": al}).head(10))


licznosci
sensor_error: 200
true_anomaly: 284

Przyklady decyzji:
  decision alert
0     keep      
1     keep      
2     keep      
3     keep      
4     keep      
5     keep      
6     keep      
7     keep      
8     keep      
9     keep      
